In [6]:
import polars as pl
from pomap.core.pomap import Pomap

In [7]:
class Model:

    def __init__(self, *args, **kwargs):
        pass

    def train(self, train_df: pl.DataFrame):
        # In practice, this would set the internal model state.
        pass

    def predict(self, test_df: pl.DataFrame):
        pass

    def validate(self, validate_df: pl.DataFrame):
        pass

In [8]:
class LiftedModel:
    def __init__(self, model_constructor: type[Model], pomap: Pomap):
        self._constructor = model_constructor
        self.pomap = pomap
        self.models = {}

    def fit(self, train_df: pl.DataFrame):

        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_train(train_df, label=label)
            model = self._constructor().train(sub)

            self.models[label] = model

    def predict(self, test_df: pl.DataFrame):
        test_df = test_df.with_row_index(name='__row_index')

        subs = []
        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_test(test_df, label=label)

            model = self.models[label]

            predictions = model.predict(
                sub.to_pandas()
                ).rename('prediction')

            predictions = pl.from_pandas(predictions)

            sub = sub.with_columns(predictions).select('prediction', '__row_index')
            subs.append(sub)

        test_df = test_df.join(pl.concat(subs), on='__row_index', how='left').drop('__row_index')

        return test_df


### Define a model type that makes it straightforward to test the PoMap behavor


In [ ]:
class PassThroughModel:

    def __init__(self, value):
        self.value = value

    def predict(self):
        return self.value